# Turkish Embedding Model Selection — Benchmark Comparison

**Project:** *Morphology-Aware Contrastive Fine-Tuning for Turkish Retrieval* (inzva AI Projects #10)

This notebook compares candidate encoder models on **trusted Turkish benchmarks** so the team can select:

1. the **semantic encoder** base model (to be LoRA fine-tuned on morpho-augmented triplets),
2. the **comparison baselines** for the paper/report,
3. the **morphological-channel encoder** (lightweight Turkish encoder for the dual-encoder ablation).

**Evidence sources (cited throughout):**

| Benchmark | What | Where |
|---|---|---|
| **TR-MTEB** — Baysan & Güngör, *Findings of EMNLP 2025* | 26 datasets / 6 tasks, retrieval = 11 datasets, primary metric nDCG@10 | [aclanthology.org/2025.findings-emnlp.471](https://aclanthology.org/2025.findings-emnlp.471/), [github.com/selmanbaysan/mteb_tr](https://github.com/selmanbaysan/mteb_tr), HF org [`trmteb`](https://huggingface.co/trmteb) |
| **Mizan leaderboard** (NewMind AI) | Live Turkish embedding leaderboard (retrieval / clustering / legal) | [newmindai-mizan.hf.space](https://newmindai-mizan.hf.space) |
| **Turkish-BEIR / TurkColBERT** | 5 BEIR-format Turkish retrieval datasets | [HF blog: late-interaction models for Turkish](https://huggingface.co/blog/nmmursit/late-interaction-models), [arXiv:2511.07595](https://arxiv.org/abs/2511.07595) |
| **Morphological probe** (ours) | pilot triplets from the project deck + extended suffix-category set | Section C below |

**Design (hybrid evaluation):**
- **Section A** — published scores (TR-MTEB paper + Mizan) → breadth, zero GPU cost.
- **Section B** — our own retrieval runs on 4 Turkish-BEIR datasets → first-hand nDCG@10 / Recall@K / MRR, including models *missing* from the paper (BGE-M3, TurkEmbed4Retrieval, EmbeddingGemma, Qwen3, …).
- **Section C** — morphological probe (suffix distractor triplets) → the project's core failure mode, per suffix category.
- **Section D** — efficiency profile (throughput / VRAM) → deployment reality for the dual-encoder + Qdrant stack.
- **Section E** — weighted selection matrix → the actual picks.

> **Runtime:** Colab Pro **A100/L4** recommended (~1.5–3 h full run). Proprietary APIs (e.g. `text-embedding-3-small`) appear as **cited reference rows only** — nothing here calls a paid API.


## 0 · Setup

Installs, imports, device detection. Set `SMOKE = True` for a quick end-to-end test
(2 small models, 1 dataset, capped queries) before a full run.

In [ ]:
# transformers is PINNED <5: v5 initializes models on the meta device, which leaves the
# GTE remote code's non-persistent buffers (position_ids, rope cos/sin caches) as
# uninitialized garbage -> "CUDA error: device-side assert triggered" on TurkEmbed4*/GTE.
# 4.56+ still covers EmbeddingGemma (gemma3) and Qwen3. Run this on a FRESH runtime.
%pip install -q -U "transformers>=4.56,<5" sentence-transformers datasets zeyrek pandas matplotlib "huggingface_hub[hf_xet]"
# NOTE: keep hf_xet INSTALLED (not uninstalled) -- some repos are xet-only and hard-error
# ("you need to install the hf_xet package") without it. What actually needs to happen is
# disabling its *usage*, which the HF_HUB_DISABLE_XET env var below does while the package
# stays present. xet's transfer path is what hangs at 0% on GCP/Colab (huggingface/xet-core
# #800, #850; huggingface/datasets#8129) -- uninstalling the package was the wrong fix.

In [ ]:
import gc, json, math, os, re, time
# Disables xet's *transfer* usage while keeping the hf_xet package installed (see install-cell
# comment) -- must be set before any huggingface_hub/datasets/sentence_transformers download.
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

SMOKE = os.environ.get("SMOKE", "0") == "1"  # quick test mode (CPU-friendly)

DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
          else "cpu")
CACHE_DIR = "cache_results"
os.makedirs(CACHE_DIR, exist_ok=True)

ACCENT = "#2563eb"   # single categorical hue — magnitude is carried by length, not color
GRAY = "#9ca3af"

def slug(model_id):
    return re.sub(r"[^a-zA-Z0-9]+", "_", model_id)

print(f"device={DEVICE}  smoke={SMOKE}  torch={torch.__version__}")

`google/embeddinggemma-300m` is **gated**: accept the license on its
[model page](https://huggingface.co/google/embeddinggemma-300m) once, then log in below
(skip the cell if you only run non-gated models — the runner will skip gated models when not logged in).

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()
from huggingface_hub import whoami
try:
    HF_LOGGED_IN = whoami() is not None
except Exception:
    HF_LOGGED_IN = False
print("HF login:", HF_LOGGED_IN)

## 1 · Model registry

One registry drives every section. Per model we track: HF id, size, embedding dim, **role**
(`semantic` = candidate for the fine-tuned encoder; `morph` = candidate for the lightweight
morphological channel; `reference` = cited-only, never run), the **query/passage prompts the
model was trained with** (forgetting these is the classic silent bug in embedding comparisons),
and published scores:

- `trmteb_retr` — TR-MTEB **retrieval subtask nDCG@10** (Table 2 of the paper),
- `mizan` — **Mizan leaderboard MTEB score** (snapshot: July 2026).


In [ ]:
CANDIDATES = [
    # ------------------------------------------------ semantic-encoder candidates
    dict(id="newmindai/TurkEmbed4Retrieval", short="TurkEmbed4Retrieval", params_m=305, dim=768,
         role="semantic", query_prompt="", doc_prompt="", trust_remote_code=True, gated=False,
         published=dict(trmteb_retr=None, mizan=None),
         notes="Project's planned base (GTE-multilingual backbone, MS MARCO-TR fine-tune; "
               "SciFact-TR NDCG@100 0.53 in arXiv:2511.07595)."),
    dict(id="google/embeddinggemma-300m", short="EmbeddingGemma-300m", params_m=308, dim=768,
         role="semantic", query_prompt="task: search result | query: ", doc_prompt="title: none | text: ",
         trust_remote_code=False, gated=True,
         published=dict(trmteb_retr=None, mizan=65.42),
         notes="#1 on Mizan; gated license."),
    dict(id="BAAI/bge-m3", short="BGE-M3", params_m=567, dim=1024,
         role="semantic", query_prompt="", doc_prompt="", trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=None, mizan=62.87),
         notes="Planned baseline; failed the 20/20 morphological pilot -> negative control."),
    dict(id="intfloat/multilingual-e5-large", short="mE5-large", params_m=560, dim=1024,
         role="semantic", query_prompt="query: ", doc_prompt="passage: ",
         trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=60.62, mizan=61.51),
         notes="Best open retrieval model in the TR-MTEB paper."),
    dict(id="intfloat/multilingual-e5-base", short="mE5-base", params_m=278, dim=768,
         role="semantic", query_prompt="query: ", doc_prompt="passage: ",
         trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=58.29, mizan=None),
         notes="Pilot negative control (failed 20/20)."),
    dict(id="Alibaba-NLP/gte-multilingual-base", short="GTE-mult-base", params_m=305, dim=768,
         role="semantic", query_prompt="", doc_prompt="", trust_remote_code=True, gated=False,
         published=dict(trmteb_retr=57.51, mizan=60.12),
         notes="Backbone family of TurkEmbed4Retrieval."),
    dict(id="ytu-ce-cosmos/turkish-e5-large", short="turkish-e5-large", params_m=560, dim=1024,
         role="semantic",
         query_prompt="Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: ",
         doc_prompt="", trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=None, mizan=60.36),
         notes="Turkish-tuned e5-instruct; verify prompt against model card."),
    dict(id="Qwen/Qwen3-Embedding-0.6B", short="Qwen3-Emb-0.6B", params_m=595, dim=1024,
         role="semantic",
         query_prompt="Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: ",
         doc_prompt="", trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=None, mizan=None),
         notes="64.33 on MMTEB multilingual (not directly comparable to TR-MTEB)."),
    dict(id="newmindai/TurkEmbed4STS", short="TurkEmbed4STS", params_m=305, dim=768,
         role="semantic", query_prompt="", doc_prompt="", trust_remote_code=True, gated=False,
         published=dict(trmteb_retr=None, mizan=62.42),
         notes="STS-tuned sibling of the planned base."),
    dict(id="trmteb/turkish-embedding-model-fine-tuned", short="trmteb-ft-110M", params_m=110, dim=768,
         role="semantic", query_prompt="", doc_prompt="", trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=52.75, mizan=60.74),
         notes="TR-MTEB authors' BERTurk-based model — small + Turkish-native."),
    # optional extras (enable manually if runtime allows)
    dict(id="magibu/embeddingmagibu-200m", short="embeddingmagibu-200m", params_m=200, dim=768,
         role="semantic", query_prompt="", doc_prompt="", trust_remote_code=True, gated=False,
         published=dict(trmteb_retr=None, mizan=59.25), enabled=False,
         notes="Turkish-distilled EmbeddingGemma (arXiv:2605.29992, TR-MTEB avg 63.9); uses custom "
               ".encode_query()/.encode_document() methods our shared encode() helper doesn't call -> "
               "would need model-specific plumbing. Disabled: not worth it for a mid-pack Mizan score."),

    # ------------------------------------------------ added session 2: Burak's 4 named models + research finds
    dict(id="emrecan/bert-base-turkish-cased-mean-nli-stsb-tr", short="emrecan-nli-stsb", params_m=110, dim=768,
         role="semantic", query_prompt="", doc_prompt="", trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=None, mizan=54.33),
         notes="Classic 2021-era Turkish SBERT (BERTurk + NLI/STSb), Apache-2.0. Legacy baseline: "
               "how far have newer models come."),
    dict(id="trmteb/turkish-embedding-model", short="trmteb-pretrain-110M", params_m=110, dim=768,
         role="semantic", query_prompt="", doc_prompt="", trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=52.54, mizan=None),
         notes="Contrastive-PRETRAINED-ONLY sibling of trmteb-ft-110M (same TR-MTEB paper, no "
               "supervised fine-tune). Ablation pair: isolates what fine-tuning adds."),
    dict(id="Lajavaness/bilingual-embedding-large", short="Lajavaness-bi-large", params_m=560, dim=1024,
         role="semantic", query_prompt="", doc_prompt="", trust_remote_code=True, gated=False,
         published=dict(trmteb_retr=None, mizan=62.47),
         notes="XLM-RoBERTa-large, trained ONLY on EN/FR (SNLI/XNLI/STSB) -> zero Turkish training data "
               "yet ranks #4 on Turkish Mizan via cross-lingual transfer. params_m is an estimate "
               "(XLM-R-large scale, not stated on model card). Morphological transfer unverified."),
    dict(id="nomic-ai/nomic-embed-text-v2-moe", short="nomic-v2-moe", params_m=475, dim=768,
         role="semantic", query_prompt="search_query: ", doc_prompt="search_document: ",
         trust_remote_code=True, gated=False,
         published=dict(trmteb_retr=None, mizan=59.54),
         notes="First general-purpose MoE embedding model (305M active of 475M total, Matryoshka to "
               "256d). Efficiency story directly relevant to the dual-encoder's compute budget."),
    dict(id="newmindai/Mursit-Base-TR-Retrieval", short="Mursit-Base", params_m=155, dim=768,
         role="semantic", query_prompt="", doc_prompt="", trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=None, mizan=55.86),
         notes="ModernBERT-base, 112.7B-token Turkish-dominant pretrain + MS MARCO-TR fine-tune. "
               "Likely the Mecellem paper's (arXiv:2601.16018) headline '155M matches 307-567M "
               "references' result -> potentially the strongest small LoRA base in the pool."),
    dict(id="newmindai/modernbert-base-tr-uncased-allnli-stsb", short="modernbert-tr-allnli", params_m=135, dim=768,
         role="semantic", query_prompt="", doc_prompt="", trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=None, mizan=56.32),
         notes="STS/NLI-tuned sibling of artiwise-ai/modernbert-base-tr-uncased (below, morph role) -> "
               "second pretrain-vs-finetune ablation pair. Native max_seq 256 (shorter; respected "
               "automatically by load_model's min(...,512) cap)."),
    dict(id="newmindai/Mursit-Large-TR-Retrieval", short="Mursit-Large", params_m=560, dim=1024,
         role="semantic", query_prompt="", doc_prompt="", trust_remote_code=True, gated=False,
         published=dict(trmteb_retr=None, mizan=56.43),
         notes="Re-enabled: same Mecellem paper family as Mursit-Base, now confirmed via arXiv:2601.16018."),

    # ------------------------------------------------ morphological-channel candidates
    dict(id="dbmdz/bert-base-turkish-cased", short="BERTurk-base-cased", params_m=110, dim=768,
         role="morph", query_prompt="", doc_prompt="", trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=None, mizan=None),
         notes="BERTurk; mean-pooled raw encoder (no contrastive training)."),
    dict(id="dbmdz/bert-base-turkish-uncased", short="BERTurk-base-uncased", params_m=110, dim=768,
         role="morph", query_prompt="", doc_prompt="", trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=34.82, mizan=None),
         notes="TR-MTEB paper's monolingual baseline (retrieval 34.82 raw)."),
    dict(id="dbmdz/distilbert-base-turkish-cased", short="DistilBERTurk", params_m=66, dim=768,
         role="morph", query_prompt="", doc_prompt="", trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=None, mizan=None), notes="Distilled BERTurk — light."),
    dict(id="ytu-ce-cosmos/turkish-base-bert-uncased", short="cosmos-base-bert", params_m=110, dim=768,
         role="morph", query_prompt="", doc_prompt="", trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=None, mizan=None), notes="Cosmos BERT (75GB corpus)."),
    dict(id="ytu-ce-cosmos/turkish-small-bert-uncased", short="cosmos-small-bert", params_m=29, dim=512,
         role="morph", query_prompt="", doc_prompt="", trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=None, mizan=None), notes="Small — attractive for the morph channel."),
    dict(id="ytu-ce-cosmos/turkish-tiny-bert-uncased", short="cosmos-tiny-bert", params_m=5, dim=128,
         role="morph", query_prompt="", doc_prompt="", trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=None, mizan=None), notes="Tiny — cheapest option."),
    dict(id="artiwise-ai/modernbert-base-tr-uncased", short="ModernBERT-TR", params_m=135, dim=768,
         role="morph", query_prompt="", doc_prompt="", trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=None, mizan=None), notes="Turkish ModernBERT, 8k context."),
    dict(id="boun-tabilab/TabiBERT", short="TabiBERT", params_m=149, dim=768,
         role="morph", query_prompt="", doc_prompt="", trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=None, mizan=None),
         notes="Dec-2025 ModernBERT-TR foundation model, 1T-token pretrain (largest-scale Turkish "
               "encoder pretrain to date), 8192 native ctx. Raw MLM, no pooling head -> same situation "
               "as the BERTurk/cosmos entries above; SentenceTransformer auto-wraps with mean pooling. "
               "Expect BERTurk-like raw scores (no retrieval/STS tuning yet)."),

    # ------------------------------------------------ cited-only references (never run)
    dict(id="openai/text-embedding-3-small", short="text-emb-3-small (API)", params_m=None, dim=1536,
         role="reference", query_prompt="", doc_prompt="", trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=64.99, mizan=None),
         notes="Proprietary upper bound in the TR-MTEB paper; not fine-tunable -> not selectable."),
    dict(id="sentence-transformers/paraphrase-multilingual-mpnet-base-v2", short="para-mpnet-v2",
         params_m=278, dim=768, role="reference", query_prompt="", doc_prompt="",
         trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=49.27, mizan=None), notes="Legacy multilingual SBERT reference."),
    dict(id="sentence-transformers/LaBSE", short="LaBSE", params_m=471, dim=768,
         role="reference", query_prompt="", doc_prompt="", trust_remote_code=False, gated=False,
         published=dict(trmteb_retr=46.47, mizan=None), notes="Bitext-oriented reference."),
]
for c in CANDIDATES:
    c.setdefault("enabled", True)

SMOKE_MODELS = {"trmteb/turkish-embedding-model-fine-tuned", "dbmdz/distilbert-base-turkish-cased",
                "trmteb/turkish-embedding-model", "boun-tabilab/TabiBERT"}

def enabled(role=None):
    out = []
    for c in CANDIDATES:
        if not c["enabled"] or c["role"] == "reference":
            continue
        if role and c["role"] != role:
            continue
        if SMOKE and c["id"] not in SMOKE_MODELS:
            continue
        if c["gated"] and not HF_LOGGED_IN:
            print(f"[skip] {c['id']} is gated and no HF login found")
            continue
        out.append(c)
    return out

print(f"{len(enabled('semantic'))} semantic / {len(enabled('morph'))} morph candidates active")

In [ ]:
import shutil
from pathlib import Path
from huggingface_hub.constants import HF_HUB_CACHE
from sentence_transformers import SentenceTransformer

def _sanity_check(model, cand):
    """Tripwire for corrupted-buffer loads (e.g. transformers-v5 meta-device init vs
    GTE remote code): garbage position_ids/rope caches -> device asserts or silent junk."""
    emb = getattr(getattr(model[0], "auto_model", None), "embeddings", None)
    pid = getattr(emb, "position_ids", None)
    if pid is not None and not torch.equal(
            pid.cpu(), torch.arange(pid.numel(), dtype=pid.dtype).reshape(pid.shape)):
        raise RuntimeError(f"{cand['id']}: position_ids buffer is corrupted — "
                           "check the transformers version pin in the install cell.")
    v = model.encode(["kontrol cümlesi"], convert_to_numpy=True)
    if not np.isfinite(v).all():
        raise RuntimeError(f"{cand['id']}: NaN/inf embeddings from a probe sentence — "
                           "model loaded incorrectly, do not trust its scores.")

def _clear_hf_cache(repo_id):
    cache_dir = Path(HF_HUB_CACHE) / f"models--{repo_id.replace('/', '--')}"
    if cache_dir.exists():
        shutil.rmtree(cache_dir, ignore_errors=True)

def load_model(cand, attempts=3):
    """Retries on any failure, same rationale as the dataset loader: HF Hub 403s on this
    infra (GCP/Colab) are frequently transient/rate-limit-related, not permanent, and a
    fresh attempt often succeeds (more so once authenticated -- see setup cell). Also clears
    this repo's local cache before each retry: a failed download can leave partial/corrupted
    files that make the *next* attempt fail with a different, more confusing error than the
    original one (observed: 403 on attempt 1 -> "Unrecognized processing class" on attempt 2,
    from an incomplete tokenizer/config cache) -- stale bytes on disk, not a code bug."""
    kwargs = {}
    if DEVICE == "cuda":
        # NEVER fp16: Gemma-family models (EmbeddingGemma) overflow -> NaN embeddings.
        # bf16 has fp32's exponent range (no overflow) and is native on A100/L4;
        # fall back to fp32 on GPUs without bf16 (e.g. T4).
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
        kwargs["model_kwargs"] = {"torch_dtype": dtype}
    last_err = None
    for i in range(attempts):
        try:
            model = SentenceTransformer(cand["id"], device=DEVICE,
                                        trust_remote_code=cand["trust_remote_code"], **kwargs)
            model.max_seq_length = min(getattr(model, "max_seq_length", 512) or 512, 512)
            _sanity_check(model, cand)
            return model
        except Exception as e:
            last_err = e
            _clear_hf_cache(cand["id"])
            if i < attempts - 1:
                print(f"    [retry {i+1}/{attempts-1}] {cand['id']}: {type(e).__name__}: {str(e)[:150]}")
                time.sleep(2 ** i)
    raise last_err

def encode(model, texts, prompt="", batch_size=64):
    if prompt:
        texts = [prompt + t for t in texts]
    return model.encode(texts, batch_size=batch_size, convert_to_numpy=True,
                        normalize_embeddings=True, show_progress_bar=True)

def free(model):
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## A · Published benchmark scores (trusted, cited)

Assembled from **TR-MTEB Table 2** (retrieval nDCG@10; Baysan & Güngör 2025) and the
**Mizan leaderboard** MTEB score (NewMind AI, snapshot July 2026). Empty cells = model not
evaluated by that source — Section B fills those gaps with our own runs.

In [ ]:
pub_rows = [dict(model=c["short"], role=c["role"], params_M=c["params_m"],
                 trmteb_retrieval_ndcg10=c["published"]["trmteb_retr"],
                 mizan_mteb=c["published"]["mizan"], notes=c["notes"])
            for c in CANDIDATES]
pub_df = (pd.DataFrame(pub_rows)
            .sort_values(["trmteb_retrieval_ndcg10", "mizan_mteb"], ascending=False, na_position="last")
            .reset_index(drop=True))
pub_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, col, title in [
        (axes[0], "trmteb_retrieval_ndcg10", "TR-MTEB retrieval nDCG@10 (published)"),
        (axes[1], "mizan_mteb", "Mizan leaderboard MTEB score (published)")]:
    d = pub_df.dropna(subset=[col]).sort_values(col)
    colors = [GRAY if r == "reference" else ACCENT for r in d["role"]]
    bars = ax.barh(d["model"], d[col], color=colors, height=0.62)
    ax.bar_label(bars, fmt="%.1f", padding=3, fontsize=9)
    ax.set_title(title, fontsize=11, loc="left")
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlim(0, max(d[col]) * 1.12)
    ax.grid(axis="x", color="#e5e7eb", linewidth=0.7)
    ax.set_axisbelow(True)
fig.suptitle("Gray = cited-only reference (not selectable)", fontsize=9, x=0.99, ha="right", y=0.02)
plt.tight_layout()
plt.show()

## B · Our own retrieval evaluation (Turkish-BEIR subset)

First-hand, reproducible numbers on 4 BEIR-format Turkish datasets (the small/mid ones — a
deliberate subset that fits a Colab session; MS MARCO-TR / Quora-TR are too large here):

| Dataset | HF repo | Queries (test) | Corpus |
|---|---|---|---|
| SciFact-TR | `AbdulkaderSaoud/scifact-tr` (qrels: `BeIR/scifact-qrels`) | 300 | 5.2K |
| NFCorpus-TR | `trmteb/nfcorpus-tr` | ~320 | 3.6K |
| ArguAna-TR | `trmteb/arguana-tr` | 1.4K | 8.7K |
| FiQA-TR | `selmanbaysan/fiqa-tr` | 648 | 57.6K |

Metrics: **nDCG@10** (primary, MTEB convention), **Recall@10/100**, **MRR@10**.
ArguAna convention: the query's own document is removed from its ranking (as in MTEB/BEIR).

In [ ]:
# NOTE: we deliberately do NOT use datasets.load_dataset() here. hf_xet's transfer path hangs
# at 0% on GCP/Colab, and even bypassing it, HF's resolve endpoint still redirects Xet-backed
# files through a "xet-bridge" CDN server-side (unavoidable regardless of client) -- which has
# its own GCP-specific flakiness (huggingface/xet-core#800: stalls/403s on GCP origins). No
# client-side code can avoid that redirect; the mitigation is retrying + preferring the
# official, authenticated hf_hub_download() (built by the team that runs the bridge, so more
# likely to have retry/negotiation logic for its own infra) with a raw-requests fallback.
#
# Two non-obvious things found by testing against the real repos (not guessed): (1) repos
# without native parquet on `main` (selmanbaysan/fiqa-tr, BeIR/scifact-qrels) only publish
# auto-converted parquet on the special `refs/convert/parquet` ref -- listing `main` alone
# silently returns zero files. (2) that ref name contains literal slashes and must be
# percent-encoded in the resolve URL -- hf_hub_url() handles this, hand-built f-strings 404.
#
# File PATHS are discovered at runtime (not hardcoded) because HF's auto-export layout is
# inconsistent across repos and even within one repo: trmteb/nfcorpus-tr's "default" config
# (qrels) lives under a "data/" folder while selmanbaysan/fiqa-tr's lives under "default/".
import io
import requests
from huggingface_hub import HfApi, hf_hub_url, hf_hub_download

_hf_api = HfApi()
_parquet_file_cache = {}

def _list_parquet_files(repo_id):
    """Returns (parquet_file_paths, revision_they_live_on)."""
    if repo_id not in _parquet_file_cache:
        for rev in (None, "refs/convert/parquet"):  # some repos only have auto-converted parquet here
            try:
                files = [f for f in _hf_api.list_repo_files(repo_id, repo_type="dataset", revision=rev)
                        if f.endswith(".parquet")]
            except Exception:
                files = []
            if files:
                _parquet_file_cache[repo_id] = (files, rev)
                break
        else:
            _parquet_file_cache[repo_id] = ([], None)
    return _parquet_file_cache[repo_id]

def _find_parquet_files(repo_id, config, split):
    files, rev = _list_parquet_files(repo_id)
    term = split or config
    candidates = []
    if config:
        candidates = [f for f in files if f.startswith(f"{config}/") and term in f]
        if not candidates:
            candidates = [f for f in files if f.startswith("data/") and term in f]
    if not candidates:
        candidates = [f for f in files if term in f]
    if not candidates:
        raise FileNotFoundError(f"no parquet match in {repo_id} for config={config!r} split={split!r} "
                                f"(saw {len(files)} parquet files, e.g. {files[:5]})")
    return sorted(candidates), rev

def _read_hf_parquet(repo_id, path, rev, attempts=4):
    """hf_hub_download first (authenticated, has HF's own bridge retry/negotiation logic and
    disk cache); raw requests as fallback. Retries the whole resolve+fetch each round -- the
    presigned redirect target is regenerated fresh server-side each hit, so a retry can land
    on a different, healthy CDN edge rather than replaying the same dead link."""
    last_err = None
    for i in range(attempts):
        try:
            local_path = hf_hub_download(repo_id, path, repo_type="dataset", revision=rev)
            return pd.read_parquet(local_path)
        except Exception as e1:
            last_err = e1
            try:
                url = hf_hub_url(repo_id, path, repo_type="dataset", revision=rev)
                r = requests.get(url, timeout=120)
                r.raise_for_status()
                return pd.read_parquet(io.BytesIO(r.content))
            except Exception as e2:
                last_err = e2
        if i < attempts - 1:
            print(f"    [retry {i+1}/{attempts-1}] {repo_id}/{path}: {type(last_err).__name__}: "
                  f"{str(last_err)[:120]}")
            time.sleep(2 ** i)
    raise last_err

def _read_hf_config(repo_id, config, split=None):
    paths, rev = _find_parquet_files(repo_id, config, split)
    frames = [_read_hf_parquet(repo_id, p, rev) for p in paths]
    return pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]

BEIR_DATASETS = {
    "scifact-tr":  dict(style="single", repo="AbdulkaderSaoud/scifact-tr",
                        qrels_repo="BeIR/scifact-qrels", qrels_split="test"),
    "nfcorpus-tr": dict(style="trmteb", repo="trmteb/nfcorpus-tr", qrels_split="test"),
    "arguana-tr":  dict(style="trmteb", repo="trmteb/arguana-tr", qrels_split="test"),
    "fiqa-tr":     dict(style="trmteb", repo="selmanbaysan/fiqa-tr", qrels_split="test"),
}
EVAL_DATASETS = ["nfcorpus-tr"] if SMOKE else list(BEIR_DATASETS)

def _doc_text(row):
    title = (row.get("title") or "").strip()
    text = (row.get("text") or "").strip()
    return f"{title} {text}".strip() if title else text

def load_beir_dataset(name):
    spec = BEIR_DATASETS[name]
    if spec["style"] == "single":
        corpus_rows = _read_hf_config(spec["repo"], "corpus").to_dict("records")
        query_rows = _read_hf_config(spec["repo"], "queries").to_dict("records")
        qrels_rows = _read_hf_config(spec["qrels_repo"], None, spec["qrels_split"]).to_dict("records")
    else:  # trmteb style: configs corpus / queries / default(qrels)
        corpus_rows = _read_hf_config(spec["repo"], "corpus").to_dict("records")
        query_rows = _read_hf_config(spec["repo"], "queries").to_dict("records")
        qrels_rows = _read_hf_config(spec["repo"], "default", spec["qrels_split"]).to_dict("records")

    corpus = {str(r["_id"]): _doc_text(r) for r in corpus_rows}
    queries = {str(r["_id"]): r["text"] for r in query_rows}
    qrels = {}
    for r in qrels_rows:
        qrels.setdefault(str(r["query-id"]), {})[str(r["corpus-id"])] = float(r["score"])
    queries = {qid: q for qid, q in queries.items() if qid in qrels}  # BEIR convention

    if SMOKE:  # tiny but valid subset: keep gold docs + a filler sample
        qids = list(queries)[:16]
        queries = {q: queries[q] for q in qids}
        qrels = {q: qrels[q] for q in qids}
        gold = {d for q in qids for d in qrels[q]}
        filler = [d for d in corpus if d not in gold][:1200]
        corpus = {d: corpus[d] for d in list(gold) + filler}
    return corpus, queries, qrels

DATA = {name: load_beir_dataset(name) for name in EVAL_DATASETS}
for name, (c, q, r) in DATA.items():
    print(f"{name}: corpus={len(c)}  queries={len(q)}  qrels={len(r)}")

In [ ]:
def dcg(gains):
    return sum(g / math.log2(i + 2) for i, g in enumerate(gains))

def evaluate_run(qrels, run, k_ndcg=10, k_mrr=10, ks_recall=(10, 100)):
    """qrels: {qid: {did: score}}; run: {qid: ranked list of dids}"""
    ndcgs, mrrs = [], []
    recalls = {k: [] for k in ks_recall}
    for qid, ranked in run.items():
        rel = qrels.get(qid, {})
        if not rel:
            continue
        ideal = dcg(sorted(rel.values(), reverse=True)[:k_ndcg])
        ndcgs.append(dcg([rel.get(d, 0.0) for d in ranked[:k_ndcg]]) / ideal if ideal > 0 else 0.0)
        rr = next((1.0 / (i + 1) for i, d in enumerate(ranked[:k_mrr]) if rel.get(d, 0) > 0), 0.0)
        mrrs.append(rr)
        n_rel = sum(1 for v in rel.values() if v > 0)
        for k in ks_recall:
            recalls[k].append(sum(1 for d in ranked[:k] if rel.get(d, 0) > 0) / n_rel if n_rel else 0.0)
    out = {"nDCG@10": np.mean(ndcgs), "MRR@10": np.mean(mrrs)}
    out.update({f"Recall@{k}": np.mean(recalls[k]) for k in ks_recall})
    return {m: round(float(v) * 100, 2) for m, v in out.items()}

def search(query_emb, doc_emb, doc_ids, query_ids, top_k=100, remove_self=False):
    """Cosine top-k via chunked matmul (embeddings are L2-normalized)."""
    dev = "cuda" if DEVICE == "cuda" else "cpu"
    D = torch.from_numpy(doc_emb).to(dev)
    Q = torch.from_numpy(query_emb).to(dev)
    run = {}
    for start in range(0, len(Q), 256):
        chunk = Q[start:start + 256]
        scores = chunk @ D.T
        k = min(top_k + (1 if remove_self else 0), len(doc_ids))
        top = torch.topk(scores, k=k, dim=1).indices.cpu().numpy()
        for row, qi in zip(top, range(start, start + len(chunk))):
            qid = query_ids[qi]
            ranked = [doc_ids[j] for j in row]
            if remove_self:
                ranked = [d for d in ranked if d != qid]
            run[qid] = ranked[:top_k]
    del D, Q
    return run

In [ ]:
retrieval_rows = []
for cand in enabled("semantic"):
    res_path = os.path.join(CACHE_DIR, f"retrieval_{slug(cand['id'])}{'_smoke' if SMOKE else ''}.json")
    if os.path.exists(res_path):
        retrieval_rows += json.load(open(res_path))
        print(f"[cache] {cand['short']}")
        continue
    print(f"\n=== {cand['id']} ===")
    try:
        model = load_model(cand)
    except Exception as e:  # gated 403, OOM, remote-code break -> skip, don't abort the run
        print(f"  [skip] could not load {cand['id']}: {type(e).__name__}: {str(e)[:160]}")
        continue
    try:
        rows = []
        for name in EVAL_DATASETS:
            corpus, queries, qrels = DATA[name]
            doc_ids, query_ids = list(corpus), list(queries)
            t0 = time.time()
            doc_emb = encode(model, [corpus[d] for d in doc_ids], cand["doc_prompt"])
            query_emb = encode(model, [queries[q] for q in query_ids], cand["query_prompt"])
            run = search(query_emb, doc_emb, doc_ids, query_ids, remove_self=(name == "arguana-tr"))
            metrics = evaluate_run(qrels, run)
            rows.append(dict(model=cand["short"], dataset=name, **metrics,
                             encode_sec=round(time.time() - t0, 1)))
            print(f"  {name}: {metrics}")
        json.dump(rows, open(res_path, "w"))
        retrieval_rows += rows
    except Exception as e:
        print(f"  [skip] error evaluating {cand['id']}: {type(e).__name__}: {str(e)[:160]}")
    finally:
        free(model)

retr_df = pd.DataFrame(retrieval_rows)
retr_df

In [ ]:
pivot = retr_df.pivot_table(index="model", columns="dataset", values="nDCG@10")
pivot["mean"] = pivot.mean(axis=1)
pivot = pivot.sort_values("mean", ascending=False).round(2)
display(pivot)

d = pivot["mean"].sort_values()
fig, ax = plt.subplots(figsize=(8, 0.45 * len(d) + 1.2))
bars = ax.barh(d.index, d.values, color=ACCENT, height=0.62)
ax.bar_label(bars, fmt="%.1f", padding=3, fontsize=9)
ax.set_title(f"Our runs — mean nDCG@10 across {len(EVAL_DATASETS)} Turkish-BEIR datasets",
             fontsize=11, loc="left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#e5e7eb", linewidth=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

**Sanity check vs published numbers** (do this before trusting the table):

- TurkEmbed4Retrieval reported **NDCG@100 ≈ 0.53 on SciFact-TR** ([arXiv:2511.07595](https://arxiv.org/abs/2511.07595)) — our SciFact-TR number should land in that ballpark (nDCG@10 is typically a bit lower than @100).
- Model *ordering* of the e5 family should match the TR-MTEB paper (large > base).
- If a model looks catastrophically bad (< 20 nDCG@10), suspect a **prompt/pooling bug**, not a bad model — check the registry prompts first.

## C · Morphological probe — the project's core failure mode

Reproduces the pilot study (project deck p.8) and extends it to **50 triplets** balanced over
5 suffix categories: *hal eki* (case/direction), *olumsuzluk* (negation), *yeterlilik*
(ability, negative), *ettirgen* (causative), *zaman* (tense).

Each triplet: **anchor** (query), **distractor** (same root, suffix flips the meaning — the
*wrong* memory), **positive** (true paraphrase, different surface form). We embed with each
model's *retrieval* protocol (query prompt on anchor, doc prompt on candidates — exactly how
an agent-memory lookup would run) and score:

- **accuracy** = share of triplets with `sim(anchor, positive) > sim(anchor, distractor)`
- **margin** = mean `sim(a,p) − sim(a,d)` (how much headroom the fine-tuning must create)

A pretrained model that already scores high here needs less morphological repair; the small
`morph`-role encoders are probed too, as starting points for the morphological channel.

In [ ]:
PILOT_TRIPLETS = [  # verbatim from the project pilot (deck p.8)
    dict(anchor="Son dersten sonra hemen okuldan eve geldim.", distractor="Son dersten sonra okula geldim.",
         positive="Dersler bitince doğruca eve döndüm.", category="hal eki"),
    dict(anchor="Dün sabahki proje toplantısına katıldım.", distractor="Dün sabahki proje toplantısına katılmadım.",
         positive="Dünkü görüşmede ben de vardım.", category="olumsuzluk"),
    dict(anchor="Bu ayki elektrik faturasını dün ödedim.", distractor="Bu ayki elektrik faturasını dün ödeyemedim.",
         positive="Elektrik borcumu dün akşam tamamen kapattım.", category="yeterlilik"),
    dict(anchor="Uzun bir aradan sonra köyden şehre döndüm.", distractor="Uzun bir aradan sonra köye döndüm.",
         positive="Ziyaretim bitti, artık şehre geri geldim.", category="hal eki"),
    dict(anchor="Siparişim öğlen saatlerinde kargoya verildi.", distractor="Siparişim öğlen saatlerinde kargoya verilecek.",
         positive="Paketim öğlen civarında yola çıktı.", category="zaman"),
    dict(anchor="Salondaki kapıyı gelir gelmez açtım.", distractor="Salondaki kapıyı gelir gelmez açtırdım.",
         positive="Gelir gelmez salonun girişini kendim araladım.", category="ettirgen"),
    dict(anchor="Geçen ay ailemle birlikte şehre taşındım.", distractor="Geçen ay ailemle birlikte şehirden taşındım.",
         positive="Geçen ay yeni bir kente yerleştim.", category="hal eki"),
    dict(anchor="Bankadaki vadeli hesabımı geçen hafta kapattım.", distractor="Geçen hafta vadeli hesabımı kapatmadım.",
         positive="Vadeli mevduatımı geçen hafta bozdurup sonlandırdım.", category="olumsuzluk"),
    dict(anchor="Doktorun verdiği ilacı yemekten sonra içtim.", distractor="Doktorun verdiği ilacı yemekten sonra içirdim.",
         positive="Doktorun verdiği reçetedeki hapımı yemeğin ardından aldım.", category="ettirgen"),
    dict(anchor="Dönem sonundaki matematik sınavını geçtim.", distractor="Dönem sonundaki matematik sınavını geçemedim.",
         positive="Matematik finalinden geçerli bir not aldım.", category="yeterlilik"),
]

In [ ]:
EXTENDED_TRIPLETS = [
    # ---- hal eki (case/direction) — same verb, dative<->ablative flip
    dict(anchor="Geçen hafta yeni ofise taşındık.", distractor="Geçen hafta yeni ofisten taşındık.",
         positive="Ekip olarak yeni çalışma alanımıza geçtik.", category="hal eki"),
    dict(anchor="Kediyi eve aldık.", distractor="Kediyi evden aldık.",
         positive="Kediyi sahiplenip yanımıza getirdik.", category="hal eki"),
    dict(anchor="Paket şubeye gönderildi.", distractor="Paket şubeden gönderildi.",
         positive="Kargo ilgili şube adresine yollandı.", category="hal eki"),
    dict(anchor="Öğrenciler sınıfa alındı.", distractor="Öğrenciler sınıftan alındı.",
         positive="Öğrencilerin derslik girişine izin verildi.", category="hal eki"),
    dict(anchor="Ali İstanbul ofisine atandı.", distractor="Ali İstanbul ofisinden atandı.",
         positive="Ali'nin yeni görev yeri İstanbul ofisi oldu.", category="hal eki"),
    dict(anchor="Misafirler köyden geldi.", distractor="Misafirler köye geldi.",
         positive="Konuklarımız kırsaldan yola çıkıp bize ulaştı.", category="hal eki"),
    dict(anchor="Toplantı notlarını yeni sisteme aktardım.", distractor="Toplantı notlarını yeni sistemden aktardım.",
         positive="Kayıtları güncel platforma taşıdım.", category="hal eki"),
    dict(anchor="Babam yurtdışından döndü.", distractor="Babam yurtdışına döndü.",
         positive="Babam yabancı ülkedeki hayatını bitirip geri geldi.", category="hal eki"),
    # ---- olumsuzluk (negation)
    dict(anchor="Sabahki e-postayı okudum.", distractor="Sabahki e-postayı okumadım.",
         positive="Gelen maili sabah inceledim.", category="olumsuzluk"),
    dict(anchor="Kirayı bu ay zamanında yatırdım.", distractor="Kirayı bu ay zamanında yatırmadım.",
         positive="Bu ayki kira ödemesini gününde yaptım.", category="olumsuzluk"),
    dict(anchor="Doktorun önerdiği diyete uydum.", distractor="Doktorun önerdiği diyete uymadım.",
         positive="Hekimin verdiği beslenme programını harfiyen uyguladım.", category="olumsuzluk"),
    dict(anchor="Yeni telefonu satın aldım.", distractor="Yeni telefonu satın almadım.",
         positive="Yeni cihazı sonunda kendime edindim.", category="olumsuzluk"),
    dict(anchor="Toplantı davetini kabul ettim.", distractor="Toplantı davetini kabul etmedim.",
         positive="Görüşme çağrısına olumlu yanıt verdim.", category="olumsuzluk"),
    dict(anchor="Sözleşmeyi dün imzaladım.", distractor="Sözleşmeyi dün imzalamadım.",
         positive="Anlaşma evrakını dün onaylayıp mühürledim.", category="olumsuzluk"),
    dict(anchor="Akşam ilacımı içtim.", distractor="Akşam ilacımı içmedim.",
         positive="Gece dozumu aksatmadan aldım.", category="olumsuzluk"),
    dict(anchor="Projeyi teslim tarihinde bitirdim.", distractor="Projeyi teslim tarihinde bitirmedim.",
         positive="Çalışmayı son güne kalmadan tamamladım.", category="olumsuzluk"),
    # ---- yeterlilik (ability, negative)
    dict(anchor="Sunumu zamanında yetiştirebildim.", distractor="Sunumu zamanında yetiştiremedim.",
         positive="Sunum dosyam tam vaktinde hazırdı.", category="yeterlilik"),
    dict(anchor="Sabah erken kalkabildim.", distractor="Sabah erken kalkamadım.",
         positive="Bu sabah alarmımla birlikte uyandım.", category="yeterlilik"),
    dict(anchor="Sorunun cevabını hatırlayabildim.", distractor="Sorunun cevabını hatırlayamadım.",
         positive="Yanıt aklıma hemen geldi.", category="yeterlilik"),
    dict(anchor="Konsere bilet bulabildim.", distractor="Konsere bilet bulamadım.",
         positive="Konser için yer ayırtmayı başardım.", category="yeterlilik"),
    dict(anchor="Krediyi zamanında ödeyebildim.", distractor="Krediyi zamanında ödeyemedim.",
         positive="Borç taksitimi vadesinde kapattım.", category="yeterlilik"),
    dict(anchor="Uçağa son anda yetişebildim.", distractor="Uçağa son anda yetişemedim.",
         positive="Kalkıştan hemen önce uçağa bindim.", category="yeterlilik"),
    dict(anchor="Dosyayı sunucuya yükleyebildim.", distractor="Dosyayı sunucuya yükleyemedim.",
         positive="Belgeyi sisteme aktarmayı başardım.", category="yeterlilik"),
    dict(anchor="Yöneticiyle görüşebildim.", distractor="Yöneticiyle görüşemedim.",
         positive="Müdürle yüz yüze konuşma fırsatı buldum.", category="yeterlilik"),
    # ---- ettirgen (causative)
    dict(anchor="Arabayı yıkadım.", distractor="Arabayı yıkattım.",
         positive="Aracımı kendim temizledim.", category="ettirgen"),
    dict(anchor="Saçımı kestim.", distractor="Saçımı kestirdim.",
         positive="Saçlarımı kendi elimle kısalttım.", category="ettirgen"),
    dict(anchor="Evi boyadım.", distractor="Evi boyattım.",
         positive="Duvarları bizzat kendim renklendirdim.", category="ettirgen"),
    dict(anchor="Raporu yazdım.", distractor="Raporu yazdırdım.",
         positive="Belgeyi kendim kaleme aldım.", category="ettirgen"),
    dict(anchor="Düğün pastasını yaptım.", distractor="Düğün pastasını yaptırdım.",
         positive="Pastayı kendi ellerimle hazırladım.", category="ettirgen"),
    dict(anchor="Paketi taşıdım.", distractor="Paketi taşıttım.",
         positive="Koliyi kendim sırtlanıp götürdüm.", category="ettirgen"),
    dict(anchor="Makaleyi çevirdim.", distractor="Makaleyi çevirttim.",
         positive="Metnin tercümesini kendim yaptım.", category="ettirgen"),
    dict(anchor="Bahçedeki çimleri biçtim.", distractor="Bahçedeki çimleri biçtirdim.",
         positive="Çimenleri kendim keserek kısalttım.", category="ettirgen"),
    # ---- zaman (tense)
    dict(anchor="Siparişi kargoya verdim.", distractor="Siparişi kargoya vereceğim.",
         positive="Paket şu an yolda.", category="zaman"),
    dict(anchor="Faturayı ödedim.", distractor="Faturayı ödeyeceğim.",
         positive="Ödemeyi tamamladım, borç kapandı.", category="zaman"),
    dict(anchor="Toplantı başladı.", distractor="Toplantı başlayacak.",
         positive="Görüşme şu anda devam ediyor.", category="zaman"),
    dict(anchor="Uçak İstanbul'a indi.", distractor="Uçak İstanbul'a inecek.",
         positive="Uçuş sona erdi, yolcular terminale geçti.", category="zaman"),
    dict(anchor="Yeni şubemiz açıldı.", distractor="Yeni şubemiz açılacak.",
         positive="Mağazamızın yeni noktası hizmete girdi.", category="zaman"),
    dict(anchor="Maaşlar yattı.", distractor="Maaşlar yatacak.",
         positive="Ücret ödemeleri hesaplara geçti.", category="zaman"),
    dict(anchor="Sınav sonuçları açıklandı.", distractor="Sınav sonuçları açıklanacak.",
         positive="Notlar sistemde görülebiliyor.", category="zaman"),
    dict(anchor="Sözleşme imzalandı.", distractor="Sözleşme imzalanacak.",
         positive="Anlaşma resmiyet kazandı.", category="zaman"),
]

MORPH_TRIPLETS = PILOT_TRIPLETS + EXTENDED_TRIPLETS
print(f"{len(MORPH_TRIPLETS)} triplets:", pd.Series([t["category"] for t in MORPH_TRIPLETS]).value_counts().to_dict())
json.dump(MORPH_TRIPLETS, open("morph_probe_triplets.json", "w"), ensure_ascii=False, indent=1)  # for team review

**Root check (zeyrek/Zemberek):** confirm each anchor–distractor pair really shares the root and
differs only morphologically. Zeyrek is an alpha port of Zemberek — treat disagreements as *review
flags*, not failures.

In [ ]:
try:
    import logging, nltk, zeyrek
    nltk.download("punkt_tab", quiet=True)  # zeyrek tokenizes via NLTK
    logging.getLogger("zeyrek").setLevel(logging.ERROR)
    analyzer = zeyrek.MorphAnalyzer()
    flags = []
    for i, t in enumerate(MORPH_TRIPLETS):
        wa, wd = t["anchor"].rstrip(".").split()[-1], t["distractor"].rstrip(".").split()[-1]
        la = {lemma for _, lemmas in analyzer.lemmatize(wa) for lemma in lemmas}
        ld = {lemma for _, lemmas in analyzer.lemmatize(wd) for lemma in lemmas}
        if la and ld and not (la & ld):
            flags.append((i, t["category"], wa, sorted(la), wd, sorted(ld)))
    print(f"{len(flags)} pairs flagged for manual review (final-word roots differ):")
    for f in flags:
        print("  ", f)
except Exception as e:
    print(f"[zeyrek unavailable -> skipping root check] {e}")

In [ ]:
probe_rows = []
for cand in enabled("semantic") + enabled("morph"):
    res_path = os.path.join(CACHE_DIR, f"probe_{slug(cand['id'])}{'_smoke' if SMOKE else ''}.json")
    if os.path.exists(res_path):
        probe_rows += json.load(open(res_path))
        print(f"[cache] {cand['short']}")
        continue
    print(f"=== {cand['id']} ===")
    try:
        model = load_model(cand)
    except Exception as e:
        print(f"  [skip] could not load {cand['id']}: {type(e).__name__}: {str(e)[:160]}")
        continue
    try:
        A = encode(model, [t["anchor"] for t in MORPH_TRIPLETS], cand["query_prompt"]).astype(np.float32)
        P = encode(model, [t["positive"] for t in MORPH_TRIPLETS], cand["doc_prompt"]).astype(np.float32)
        D = encode(model, [t["distractor"] for t in MORPH_TRIPLETS], cand["doc_prompt"]).astype(np.float32)
        sim_p, sim_d = (A * P).sum(1), (A * D).sum(1)
        rows = [dict(model=cand["short"], role=cand["role"], category=t["category"],
                     pilot=i < len(PILOT_TRIPLETS), correct=bool(sim_p[i] > sim_d[i]),
                     margin=float(sim_p[i] - sim_d[i]))
                for i, t in enumerate(MORPH_TRIPLETS)]
        json.dump(rows, open(res_path, "w"))
        probe_rows += rows
    except Exception as e:
        print(f"  [skip] error probing {cand['id']}: {type(e).__name__}: {str(e)[:160]}")
    finally:
        free(model)

probe_df = pd.DataFrame(probe_rows)
probe_summary = (probe_df.groupby(["model", "role"])
                 .agg(accuracy=("correct", "mean"), margin=("margin", "mean"),
                      pilot_acc=("correct", lambda s: s[probe_df.loc[s.index, "pilot"]].mean()))
                 .round(3).sort_values("accuracy", ascending=False))
probe_summary

In [ ]:
acc = (probe_df.pivot_table(index="model", columns="category", values="correct", aggfunc="mean")
       .loc[probe_summary.reset_index()["model"]])
fig, ax = plt.subplots(figsize=(8, 0.45 * len(acc) + 1.5))
im = ax.imshow(acc.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(acc.columns)), acc.columns, rotation=20, ha="right")
ax.set_yticks(range(len(acc.index)), acc.index)
for i in range(acc.shape[0]):
    for j in range(acc.shape[1]):
        v = acc.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                color="white" if v > 0.6 else "#111827")
ax.set_title("Morphological probe accuracy by suffix category\n(sim(anchor,positive) > sim(anchor,distractor))",
             fontsize=11, loc="left")
fig.colorbar(im, ax=ax, shrink=0.8, label="accuracy")
plt.tight_layout()
plt.show()

**Expected pattern (pilot reproduction):** `mE5-base` and `BGE-M3` should score near **0.0 on the
10 pilot triplets** (`pilot_acc` column) — that is the deck's 20/20 failure. If they instead score
high, investigate before drawing conclusions (prompting, normalization, or data-entry drift).

## D · Efficiency profile

Encode throughput and peak VRAM per model (batch 64, seq ≤ 512, ~256 realistic passages).
The dual-encoder runs **two** encoders per memory write — the morph channel must be cheap.

In [ ]:
bench_texts = None
for name in EVAL_DATASETS:  # realistic doc lengths from the first loaded corpus
    corpus = DATA[name][0]
    bench_texts = [corpus[d] for d in list(corpus)[:256]]
    break

eff_rows = []
for cand in enabled("semantic") + enabled("morph"):
    res_path = os.path.join(CACHE_DIR, f"eff_{slug(cand['id'])}{'_smoke' if SMOKE else ''}.json")
    if os.path.exists(res_path):
        eff_rows.append(json.load(open(res_path)))
        continue
    try:
        model = load_model(cand)
    except Exception as e:
        print(f"  [skip] could not load {cand['id']}: {type(e).__name__}: {str(e)[:160]}")
        continue
    try:
        encode(model, bench_texts[:32], cand["doc_prompt"])  # warmup
        if DEVICE == "cuda":
            torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        encode(model, bench_texts, cand["doc_prompt"])
        dt = time.time() - t0
        row = dict(model=cand["short"], role=cand["role"], params_M=cand["params_m"], dim=cand["dim"],
                   sents_per_sec=round(len(bench_texts) / dt, 1),
                   peak_vram_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEVICE == "cuda" else None)
        json.dump(row, open(res_path, "w"))
        eff_rows.append(row)
    except Exception as e:
        print(f"  [skip] error benchmarking {cand['id']}: {type(e).__name__}: {str(e)[:160]}")
    finally:
        free(model)

eff_df = pd.DataFrame(eff_rows).sort_values("sents_per_sec", ascending=False)
eff_df

## E · Selection matrix

Min–max-normalized weighted score over: our mean nDCG@10 (**0.40**), morph-probe accuracy
(**0.30**), published Mizan score (**0.20**), throughput (**0.10**). Adjust `WEIGHTS` to taste —
the point is a transparent, reproducible ranking, not false precision.

In [ ]:
WEIGHTS = {"own_ndcg10": 0.40, "morph_acc": 0.30, "mizan": 0.20, "speed": 0.10}

sel = pd.DataFrame({"own_ndcg10": pivot["mean"]})
sel["morph_acc"] = probe_summary.reset_index().set_index("model")["accuracy"]
sel["mizan"] = pub_df.set_index("model")["mizan_mteb"]
sel["speed"] = eff_df.set_index("model")["sents_per_sec"]
sel["params_M"] = eff_df.set_index("model")["params_M"]
sel = sel.loc[sel.index.intersection(pivot.index)]  # semantic candidates only

def minmax(s):
    s = s.astype(float)
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng > 0 else s * 0 + 0.5

norm = sel[list(WEIGHTS)].apply(minmax)
sel["weighted_score"] = sum(norm[c].fillna(norm[c].mean()) * w for c, w in WEIGHTS.items())
sel = sel.sort_values("weighted_score", ascending=False).round(3)
display(sel)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(sel["own_ndcg10"], sel["morph_acc"] * 100,
           s=sel["params_M"].fillna(300) * 0.6, color=ACCENT, alpha=0.75, edgecolors="white")
for m, r in sel.iterrows():
    ax.annotate(m, (r["own_ndcg10"], r["morph_acc"] * 100), fontsize=8,
                xytext=(5, 4), textcoords="offset points")
ax.set_xlabel("our mean nDCG@10 (Turkish-BEIR subset)")
ax.set_ylabel("morphological probe accuracy (%)")
ax.set_title("The selection trade-off — retrieval quality vs morphological sensitivity\n(marker area ∝ parameters)",
             fontsize=11, loc="left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(color="#e5e7eb", linewidth=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
morph_pick = (probe_summary.reset_index().query("role == 'morph'")
              .merge(eff_df[["model", "params_M", "sents_per_sec"]], on="model")
              .sort_values(["accuracy", "sents_per_sec"], ascending=False).round(3))
print("Morphological-channel candidates (probe accuracy is signal, params/speed are the budget):")
morph_pick

## Decision (fill in after the full A100/L4 run)

| Role | Pick | Why (cite table/figure) |
|---|---|---|
| **Semantic base (LoRA target)** | _e.g. TurkEmbed4Retrieval / EmbeddingGemma-300m_ | top of selection matrix; Sentence-Transformers-compatible; 300M class fits LoRA budget |
| **Baselines (report)** | _BGE-M3, mE5-large/base, GTE-mult-base_ | deck-planned baselines + strongest published retrieval scores |
| **Morph-channel encoder** | _e.g. cosmos-small/tiny-bert or DistilBERTurk_ | best probe-accuracy-per-parameter in `morph_pick` |

**Caveats to carry into the report**
- Turkish-BEIR sets are machine-translated (Aya-Expanse-8B pipeline, LLM-judged) — good for *ranking models*, not absolute quality claims; TR-MTEB paper, §3.1.2.
- Published columns come from different snapshots (paper: 2025; Mizan: July 2026) — never mix them into one averaged number.
- The probe is 50 synthetic triplets — decisive for *morphological sensitivity*, indicative only. The project's held-out morphological eval set (Month 1 deliverable) is the real test.

**Next steps:** freeze picks → build morpho-augmented triplets (Zemberek + LLM synthesis) →
LoRA contrastive fine-tuning (MNRL) → re-run Sections B/C on the fine-tuned model → TR-MTEB full
retrieval run + Mizan submission.